# X2C-KRHF — Kramers-Restricted Hartree-Fock with DIIS

## Kramers 定理与时间反演对称性

在无外磁场时，含时 Schrödinger 方程在时间反演 $t \to -t$ 下保持不变。对于自旋-$\frac{1}{2}$ 费米子，时间反演算符为：

$$\mathcal{T} = -i\sigma_y K_0$$

其中 $\sigma_y$ 是 Pauli 矩阵，$K_0$ 是复共轭算符。$\mathcal{T}$ 是**反幺正**（antiunitary）算符，满足 $\mathcal{T}^2 = -1$。

**Kramers 定理**：对于时间反演不变的费米子体系，每个能级至少二重简并。若 $|\psi\rangle$ 是本征态，则 $\mathcal{T}|\psi\rangle$ 也是能量相同的本征态，且 $\langle\psi|\mathcal{T}\psi\rangle = 0$（两者正交）。

**Kramers pair** $\{\phi_i, \bar{\phi}_i = \mathcal{T}\phi_i\}$ 是一对通过时间反演关联的旋量轨道。在 Hartree-Fock 层面，KRHF（Kramers-Restricted HF）强制 MO 系数保持此结构——每对占据的 Kramers pair 容纳 2 个电子。相比之下，GHF 不施加此限制：闭壳层体系会自然得到 Kramers 对称的解，但开壳层体系可能出现对称性破缺。

---

## j-adapted 旋量基 vs 自旋轨道 AO 基

| 基组 | 构造 | 使用场景 |
|------|------|----------|
| 自旋轨道 AO $\{\chi_\mu\alpha, \chi_\mu\beta\}$ | 空间 AO × 自旋的直积 | GHF, X2C-GHF |
| j-adapted 旋量基 | 通过 `sph2spinor_coeff()` 从球谐 GTO 线性组合 | KRHF, X2C-RHF |

两种基组维度相同（$n_{\text{so}} = 2n_{\text{AO}}$），但 j-adapted 基函数本身携带角动量耦合信息，因此两电子积分的 spin-block 结构不同。本 notebook 在 j-adapted 旋量基下实现 X2C-KRHF。

---

## 本 notebook 结构

1. **分子设定与积分** — 构建 j-adapted 旋量基下的 AO 积分
2. **X2C 一电子 Hamiltonian** — 使用 PySCF 内置 `x2c.get_hcore`（手工构造见 X2C-GHF notebook）
3. **Kramers 对称性与二电子项** — 时间反演矩阵、Kramers 对称化、J/K 收缩
4. **DIIS 加速** — 与 GHF 相同的 DIIS 框架
5. **SCF 迭代** — `solve_KR_FCSCE` 强制 Kramers 简并
6. **Kramers 简并性验证** — 检查轨道能量是否成对简并
7. **与 PySCF 对照**

---

## 1 · 分子设定与旋量基积分

H₂O 分子，cc-pVDZ 基组。在 j-adapted 旋量基下，AO 数量为 `nao_2c() = 2 × nao_nr()`。

关键积分：
- **重叠矩阵** `S`：`int1e_ovlp_spinor` — 旋量基下的 overlap
- **一电子 Hamiltonian** `H_core`：通过 PySCF `x2c.get_hcore` 获得 X2C 修正后的 Hamiltonian
- **两电子积分** `eri`：`int2e_spinor` — 旋量基下的 Coulomb 积分

正交化矩阵 $S^{-1/2}$ 用于 DIIS 误差向量变换。

In [ ]:
from pyscf import gto, scf
from pyscf.x2c import x2c
import numpy as np
import scipy
from scipy.linalg import fractional_matrix_power
from zquatev import solve_KR_FCSCE


mol = gto.M(atom="O 0 0 0; H 0 -0.757 0.587; H 0 0.757 0.587", basis="cc-pvdz")
nso = mol.nao_2c()
nelec = mol.nelectron
E_nn = mol.energy_nuc()

S = mol.intor_symmetric("int1e_ovlp_spinor")
A = fractional_matrix_power(S, -0.5)


H_core = x2c.get_hcore(mol)

print("S shape:", S.shape)
print("H shape:", H_core.shape)

print("Hermiticity S:", np.linalg.norm(S - S.conj().T))
print("Hermiticity H:", np.linalg.norm(H_core - H_core.conj().T))

eri = mol.intor("int2e_spinor", aosym="s1")

S shape: (48, 48)
H shape: (48, 48)
Hermiticity S: 0.0
Hermiticity H: 9.13639422357325e-15


---

## 2 · Kramers 对称性与二电子项

### 时间反演矩阵

对于 j-adapted 旋量基，PySCF 的 `mol.time_reversal_map()` 给出每个基函数在 $\mathcal{T}$ 下的映射：

- `trmaps[i] > 0`：第 $i$ 个基函数是 Kramers A 伙伴，映射到 `trmaps[i] - 1`（B 伙伴）
- `trmaps[i] < 0`：第 $i$ 个基函数是 Kramers B 伙伴

函数 `time_reversal_matrix(mol, mat)` 计算矩阵的 $\mathcal{T}$-变换：

$$\tilde{M}_{ij} = s_i s_j M^*_{\tau(i), \tau(j)}$$

其中 $\tau$ 是时间反演诱导的指标置换，$s_i = \pm 1$ 是相位因子。

### Kramers 对称化

`symmetrize_kr(mol, mat)` 对矩阵施加 Kramers 对称性约束：
1. 强制 Hermiticity：$M \leftarrow \frac{1}{2}(M + M^\dagger)$
2. 强制 $\mathcal{T}$-对称：$M \leftarrow \frac{1}{2}(M + \mathcal{T}M\mathcal{T}^{-1})$
3. 再次 Hermiticity

这确保密度矩阵和 Fock 矩阵满足 Kramers 限制的块结构：

$$M = \begin{pmatrix} A & B \\ -B^* & A^* \end{pmatrix}$$

其中 $A = A^\dagger$（Hermitian），$B = -B^T$（反对称），对应 Kramers A/B 分块。

### 初始猜测

`get_init_guess` 从非相对论 RHF 的 MINAO 密度出发，通过 `sph2spinor_coeff()` 变换矩阵投影到旋量基，再 Kramers 对称化得到合理的初始密度。

### J/K 收缩

与 GHF 相同的收缩公式，但在 j-adapted 旋量基下计算：

$$J_{pq} = \sum_{rs} (pq|rs) D_{sr}, \quad K_{pq} = \sum_{rs} (ps|rq) D_{sr}$$

注意 $D_{sr}$（而非 $D_{rs}$）的指标顺序——对于复旋量基，这是正确的收缩方式。

In [18]:
def get_occ(mo_energy, nocc):
    e_idx = np.argsort(mo_energy)
    mo_occ = np.zeros_like(mo_energy)
    mo_occ[e_idx[:nocc]] = 1
    return mo_occ


def make_rdm1(mo_coeff, mo_occ):
    mocc = mo_coeff[:, mo_occ > 0]
    occ = mo_occ[mo_occ > 0]
    return (mocc * occ).dot(mocc.conj().T)

def get_jk(eri_so, dm):
    vj = np.einsum("pqrs,sr->pq", eri_so, dm, optimize=True)
    vk = np.einsum("psrq,sr->pq", eri_so, dm, optimize=True)
    return vj, vk

def get_veff(eri_so, dm):
    vj, vk = get_jk(eri_so, dm)
    return vj - vk


def get_fock(H_core, V_eff):
    return H_core + V_eff

def energy_elec(dm, H_core, V_eff):
    e1 = np.einsum("pq,qp->", H_core, dm, optimize=True)
    e2 = 0.5 * np.einsum("pq,qp->", V_eff, dm, optimize=True)
    return (e1 + e2).real, e2.real


def energy_tot(dm, H_core, V_eff):
    return E_nn + energy_elec(dm, H_core, V_eff)[0]

def time_reversal_matrix(mol, mat):
    tao = np.asarray(mol.time_reversal_map())
    idx = np.abs(tao) - 1
    sign = np.where(tao > 0, 1.0, -1.0).astype(np.complex128)

    tmat = mat[np.ix_(idx, idx)].conj()
    tmat = sign[:, None] * tmat * sign[None, :]
    return tmat


def symmetrize_kr(mol, mat):
    mat = 0.5 * (mat + mat.conj().T)
    mat = 0.5 * (mat + time_reversal_matrix(mol, mat))
    mat = 0.5 * (mat + mat.conj().T)
    return mat


def project_dm_nr_to_spinor(mol, dm_nr):
    S_2c = mol.intor_symmetric("int1e_ovlp_spinor")
    S_nr = mol.intor_symmetric("int1e_ovlp")

    ua, ub = mol.sph2spinor_coeff()

    S_2c_nr = ua.conj().T @ S_nr + ub.conj().T @ S_nr

    P = scipy.linalg.solve(S_2c, S_2c_nr, assume_a="pos")

    dm_2c = P @ (0.5 * dm_nr) @ P.conj().T
    dm_2c = symmetrize_kr(mol, dm_2c)

    return dm_2c


def get_init_guess(mol):
    dm_nr = scf.hf.init_guess_by_minao(mol)
    return project_dm_nr_to_spinor(mol, dm_nr)



---

## 3 · DIIS 加速

DIIS（Direct Inversion in the Iterative Subspace）误差向量定义为：

$$e = S^{-1/2} (FDS - SDF) S^{-1/2}$$

即 Fock 矩阵与密度矩阵的对易子在正交基下的表示。当 SCF 收敛时 $[F, D] = 0$，误差向量归零。
其余步骤（B 矩阵构造、线性方程求解、Fock 矩阵外推）与 GHF 完全一致。

In [19]:
def get_err_vec(fock, dm, S, X):
    err = fock @ dm @ S - S @ dm @ fock
    return X.conj().T @ err @ X


def apply_diis(fock_list, err_vec_list):
    n = len(fock_list)
    dim = n + 1
    B = np.empty((dim, dim))
    B[-1, :] = -1
    B[:, -1] = -1
    B[-1, -1] = 0

    for i in range(n):
        for j in range(n):
            B[i, j] = np.vdot(err_vec_list[i], err_vec_list[j]).real

    rhs = np.zeros(dim)
    rhs[-1] = -1

    try:
        coeff = np.linalg.solve(B, rhs)[:-1]
    except np.linalg.LinAlgError:
        coeff = np.linalg.lstsq(B, rhs, rcond=None)[0][:-1]

    return np.einsum("i,ipq->pq", coeff, np.asarray(fock_list), optimize=True)

---

## 4 · X2C-KRHF + DIIS 完整运行

SCF 迭代的核心步骤：

1. **构造有效势**：$V_{\text{eff}} = J - K$
2. **构造 Fock 矩阵**：$F = H_{\text{X2C}} + V_{\text{eff}}$
3. **DIIS 外推**：从第 3 轮开始，用 DIIS 外推 Fock 矩阵
4. **Kramers 对对角化**：调用 `solve_KR_FCSCE(mol, F, S)`，它在内部做：
   - 按时间反演映射将 F 和 S 重排为 Kramers 配对顺序 $\{|A\rangle, |B\rangle\}$
   - 在 Kramers 适配基下做 Löwdin 正交化 $F_{\text{orth}} = S^{-1/2} F S^{-1/2}$
   - 调用 quaternion 对角化库，确保本征值成对简并
   - 将本征矢量映回原始 AO 排序
5. **构造新密度**：取最低 `nelec` 个旋量轨道，每个占据数为 1
6. **收敛判据**：能量变化 $< 10^{-9}$ 且密度变化 $< 10^{-7}$

对于 H₂O（10 个电子），占据 5 对 Kramers pairs（共 10 个旋量轨道），每对轨道能量严格简并。

In [20]:
max_cycle = 100
max_diis = 8
conv_tol = 1e-9
conv_tol_dm = 1e-7

e_old = 0.0
fock_list = []
err_vec_list = []

dm = get_init_guess(mol)

print(f"{'Iter':>4s}  {'E_total':>16s}  {'dE':>10s}  {'dD':>10s}")
print("-" * 46)

converged = False
for cycle in range(max_cycle):
    V_eff = get_veff(eri, dm)
    fock = get_fock(H_core, V_eff)
    e_tot = energy_tot(dm, H_core, V_eff)

    err = get_err_vec(fock, dm, S, A)
    fock_list.append(fock)
    err_vec_list.append(err)
    if len(fock_list) > max_diis:
        fock_list.pop(0)
        err_vec_list.pop(0)

    if cycle >= 2:
        fock = apply_diis(fock_list, err_vec_list)

    mo_energy, mo_coeff = solve_KR_FCSCE(mol, fock, S)
    mo_occ = get_occ(mo_energy, nelec)
    dm_new = make_rdm1(mo_coeff, mo_occ)

    e_diff = abs(e_tot - e_old)
    d_norm = np.linalg.norm(dm_new - dm)

    print(f"{cycle+1:4d}  {e_tot:16.10f}  {e_diff:10.3e}  {d_norm:10.3e}")

    if e_diff < conv_tol and d_norm < conv_tol_dm:
        converged = True
        dm = dm_new
        V_eff = get_veff(eri, dm)
        fock = get_fock(H_core, V_eff)
        e_tot = energy_tot(dm, H_core, V_eff)
        print(f"\nConverged in {cycle+1} iterations!")
        print(f"E_total (X2C-KRHF+DIIS/cc-pVDZ) = {e_tot:.10f} Hartree")
        print(f"E_elec = {e_tot - E_nn:.10f}")
        print(f"E_nn   = {E_nn:.10f}")
        break

    dm = dm_new
    e_old = e_tot

if not converged:
    print("WARNING: X2C-KRHF SCF did not converge!")

Iter           E_total          dE          dD
----------------------------------------------
   1    -75.9055034307   7.591e+01   9.228e-01
   2    -76.0361865475   1.307e-01   2.633e-01
   3    -76.0662728976   3.009e-02   9.203e-02
   4    -76.0752648036   8.992e-03   1.178e-02
   5    -76.0754166803   1.519e-04   4.615e-03
   6    -76.0754307313   1.405e-05   9.771e-04
   7    -76.0754312137   4.824e-07   1.885e-04
   8    -76.0754312261   1.241e-08   2.117e-05
   9    -76.0754312263   1.902e-10   7.974e-06
  10    -76.0754312263   1.883e-11   1.745e-06
  11    -76.0754312263   8.384e-13   4.055e-07
  12    -76.0754312263   1.279e-13   1.435e-07
  13    -76.0754312263   2.842e-14   1.743e-08

Converged in 13 iterations!
E_total (X2C-KRHF+DIIS/cc-pVDZ) = -76.0754312263 Hartree
E_elec = -85.2636896441
E_nn   = 9.1882584177


---

## 5 · Kramers 简并性验证

`solve_KR_FCSCE` 返回的轨道能量是 Kramers-paired 的：$[\epsilon_0, \epsilon_0, \epsilon_1, \epsilon_1, \ldots]$。以下验证：
1. 占据轨道与虚轨道的能量是否严格成对简并
2. MO 系数是否满足 Kramers 结构 $\bar{C} = \mathcal{T}C$
3. 密度矩阵是否满足 Kramers 块结构

In [21]:
# --- 验证 Kramers 简并 ---

# 1. 检查轨道能量是否成对简并
n_pairs = nso // 2
# solve_KR_FCSCE 返回的能量按 Kramers pair 排列: [e0, e0, e1, e1, ...]
e_pairs = mo_energy.reshape(-1, 2)
max_pair_split = np.max(np.abs(e_pairs[:, 0] - e_pairs[:, 1]))
print(f"Max Kramers pair energy split: {max_pair_split:.2e}")

# 打印占据的 Kramers pairs
print(f"\n{'Pair':>5s}  {'E_occ (A)':>14s}  {'E_occ (B)':>14s}  {'Split':>10s}")
print("-" * 47)
for i in range(nelec // 2):
    ea = mo_energy[2 * i]
    eb = mo_energy[2 * i + 1]
    print(f"{i:5d}  {ea:14.8f}  {eb:14.8f}  {abs(ea - eb):10.2e}")

# 打印前几个虚 Kramers pairs
print(f"\n{'Pair':>5s}  {'E_vir (A)':>14s}  {'E_vir (B)':>14s}  {'Split':>10s}")
print("-" * 47)
nvir_show = min(5, n_pairs - nelec // 2)
for i in range(nvir_show):
    j = nelec // 2 + i
    ea = mo_energy[2 * j]
    eb = mo_energy[2 * j + 1]
    print(f"{j:5d}  {ea:14.8f}  {eb:14.8f}  {abs(ea - eb):10.2e}")

# HOMO-LUMO gap
e_homo = max(mo_energy[:nelec])
e_lumo = min(mo_energy[nelec:])
print(f"\nHOMO: {e_homo:.8f},  LUMO: {e_lumo:.8f},  Gap: {e_lumo - e_homo:.8f}")

# 2. 验证每个 Kramers pair 的密度矩阵满足时间反演对称性
# 对每对 (C_A, C_B)，构造 D_pair = |C_A><C_A| + |C_B><C_B|，检查 T D_pair T^{-1} = D_pair
max_pair_asym = 0.0
for p in range(n_pairs):
    vA = mo_coeff[:, 2 * p]
    vB = mo_coeff[:, 2 * p + 1]
    D_pair = np.outer(vA, vA.conj()) + np.outer(vB, vB.conj())
    T_D_pair = time_reversal_matrix(mol, D_pair)
    asym = np.linalg.norm(D_pair - T_D_pair)
    max_pair_asym = max(max_pair_asym, asym)
print(f"\nMax Kramers asymmetry per pair: {max_pair_asym:.2e}  (expect ~0)")

# 3. 验证总密度矩阵的 Kramers 对称性:  T D T^{-1} = D
T_dm = time_reversal_matrix(mol, dm)
dm_kr_err = np.linalg.norm(dm - T_dm)
print(f"||D - T D T^{-1}|| = {dm_kr_err:.2e}  (expect ~0)")

# 4. Kramers 块结构（需要先将指标对齐到 Kramers 配对顺序）
#   idx2 = [A0, A1, ..., B0, B1, ...] 使得第 k 个 A 与第 k 个 B 配对
trmaps = np.asarray(mol.time_reversal_map())
idxA = np.where(trmaps > 0)[0]
idxB = trmaps[idxA] - 1  # idxB[k] = τ(idxA[k])
idx2 = np.hstack([idxA, idxB])

dm_kr = dm[np.ix_(idx2, idx2)]
nA = len(idxA)
A_blk = dm_kr[:nA, :nA]
B_blk = dm_kr[:nA, nA:]
C_blk = dm_kr[nA:, :nA]
D_blk = dm_kr[nA:, nA:]

print(f"\nKramers block structure (reordered basis):")
print(f"  ||D_blk - A_blk*|| = {np.linalg.norm(D_blk - A_blk.conj()):.2e}  (expect ~0)")
print(f"  ||C_blk + B_blk*|| = {np.linalg.norm(C_blk + B_blk.conj()):.2e}  (expect ~0)")
print(f"  ||B_blk + B_blk^T|| = {np.linalg.norm(B_blk + B_blk.T):.2e}  (expect ~0, B antisymmetric)")
print(f"  ||A_blk - A_blk^†|| = {np.linalg.norm(A_blk - A_blk.conj().T):.2e}  (expect ~0, A Hermitian)")

Max Kramers pair energy split: 0.00e+00

 Pair       E_occ (A)       E_occ (B)       Split
-----------------------------------------------
    0    -20.56941831    -20.56941831    0.00e+00
    1     -1.33794306     -1.33794306    0.00e+00
    2     -0.69853657     -0.69853657    0.00e+00
    3     -0.56664058     -0.56664058    0.00e+00
    4     -0.49304210     -0.49304210    0.00e+00

 Pair       E_vir (A)       E_vir (B)       Split
-----------------------------------------------
    5      0.18516382      0.18516382    0.00e+00
    6      0.25617239      0.25617239    0.00e+00
    7      0.78852749      0.78852749    0.00e+00
    8      0.85399229      0.85399229    0.00e+00
    9      1.16316338      1.16316338    0.00e+00

HOMO: -0.49304210,  LUMO: 0.18516382,  Gap: 0.67820592

Max Kramers asymmetry per pair: 5.15e-15  (expect ~0)
||D - T D T^-1|| = 6.18e-16  (expect ~0)

Kramers block structure (reordered basis):
  ||D_blk - A_blk*|| = 3.77e-16  (expect ~0)
  ||C_blk + B_blk*|| 

---

## 6 · 与 PySCF 对照

In [22]:
# --- 与 PySCF 参照计算比较 ---

# 1. Spin-free X2C-RHF（标量相对论，无 SOC）
mf_sfx2c = scf.RHF(mol).sfx2c1e()
mf_sfx2c.kernel()

# 2. X2C-GHF（含 SOC，无 Kramers 限制）
mf_x2c_ghf = scf.GHF(mol).x2c1e()
mf_x2c_ghf.kernel()

print(f"{'Method':<30s}  {'E_total (Hartree)':>20s}")
print("-" * 52)
print(f"{'X2C-KRHF (this work)':<30s}  {e_tot:20.10f}")
print(f"{'PySCF sfX2C-RHF':<30s}  {mf_sfx2c.e_tot:20.10f}")
print(f"{'PySCF X2C-GHF':<30s}  {mf_x2c_ghf.e_tot:20.10f}")

print(f"\nΔ(self - sfX2C) = {e_tot - mf_sfx2c.e_tot:.6e}  (SOC contribution)")
print(f"Δ(self - GHF)   = {e_tot - mf_x2c_ghf.e_tot:.2e}  (should be ~0 for closed-shell)")

# 对于闭壳层体系 (H2O)，KRHF 应给出与 GHF 相同的结果
# 因为时间反演对称性自动保证 Kramers 简并
# KRHF 的优势在于开壳层体系（奇数电子），强制 Kramers 对称性

converged SCF energy = -76.0754290778622
converged SCF energy = -76.0754312262763  <S^2> = 9.8585231e-06  2S+1 = 1.0000197
Method                             E_total (Hartree)
----------------------------------------------------
X2C-KRHF (this work)                  -76.0754312263
PySCF sfX2C-RHF                       -76.0754290779
PySCF X2C-GHF                         -76.0754312263

Δ(self - sfX2C) = -2.148472e-06  (SOC contribution)
Δ(self - GHF)   = -5.78e-11  (should be ~0 for closed-shell)


---

## 7 · 总结

| 模块 | X2C-KRHF 形式 | 与 X2C-GHF 的区别 |
|------|--------------|-------------------|
| AO 基 | j-adapted 旋量基 (`nao_2c()`) | GHF 用自旋轨道 AO 直积基 |
| 一电子项 | `x2c.get_hcore(mol)` — 旋量基下的 complex Hermitian $H_{\text{X2C}}$ | 相同（不同基表示） |
| 二电子项 | `int2e_spinor` — 旋量基直接积分；J/K 用 `sr` 收缩 | GHF 用 `int2e` 砌 spin-block |
| 对角化 | `solve_KR_FCSCE` — quaternion 对角化，强制 Kramers 简并 | 普通 `eigh(F, S)` |
| 占据方式 | 每对 Kramers pair 容纳 2 个自旋轨道 | 按能量最低原则 |
| 轨道能量 | 严格成对简并 $[\epsilon_0,\epsilon_0,\epsilon_1,\epsilon_1,\ldots]$ | 无硬性约束 |
| 适用场景 | 含 SOC 的闭壳层/开壳层（强制时间反演对称性） | 含 SOC 的通用计算 |

### Kramers 简并的核心价值

对于**奇电子体系**（如自由基、过渡金属配合物），GHF 可能自发破缺时间反演对称性，得到非物理的对称性破缺解。KRHF 通过在对角化步骤中强制 Kramers 配对的块结构：

$$M = \begin{pmatrix} A & B \\ -B^* & A^* \end{pmatrix}$$

保证每个能级严格二重简并，得到物理正确的 Kramers 基态。对于**偶电子闭壳层体系**（如本 notebook 的 H₂O），时间反演对称性自动满足，KRHF 与 GHF 给出相同结果。